# driving pestpp-mou from python: the population is yours

`pestpp-mou` runs a genetic algorithm. draw a population, run it, breed the survivors, repeat.
the exe does that whole loop for you and the only way into it is the control file.

through the api the loop comes apart, and three things become possible that the exe has no room
for:

1. **start from a population you chose** instead of the one mou drew
2. **change the offspring before they are run** - repair the infeasible ones, drop in a member
   you want tried
3. **change the generator between generations** - de while you are still exploring, sbx+pm when
   you are polishing

all three are the same idea. mou hands you the population at the points where it would
otherwise just carry on, and whatever is in it when you hand it back is what gets run.

## the test problem

`constr`, from the moea literature. two decision variables, two objectives to minimize, two
linear constraints:

```
minimize    obj_1 = dv_0
minimize    obj_2 = (1 + dv_1) / dv_0
subject to  9*dv_0 + dv_1 >= 6
            9*dv_0 - dv_1 >= 1
with        dv_0 in [0.1, 1.0],  dv_1 in [0, 5]
```

the two objectives fight each other. pushing `dv_0` down is good for `obj_1` and bad for
`obj_2`, so the answer is a front of tradeoffs rather than a point.

the constraints go in as prior information, which means they are linear in the decision
variables and can be checked in pandas without running the model. that is what makes the repair
below possible - you can tell whether a member is feasible before you spend a run on it.

the model itself is a three line python script, so the whole notebook runs in a couple of
minutes on the serial run manager.

In [ ]:
import os
import shutil
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyemu

sys.path.insert(0, os.path.join("..", "python"))
from pestpp import Mou

POP = 30                      # members in the population
workdir = "mou_master"

if os.path.exists(workdir):
    shutil.rmtree(workdir)
os.makedirs(workdir)

In [ ]:
# the model: read the two decision variables, write the two objectives
FORWARD_RUN = """import pandas as pd

dv = pd.read_csv("dv.dat", sep=r"\\s+", index_col=0, header=None,
                 names=["parnme", "parval1"]).parval1

with open("obj.dat", "w") as f:
    f.write("obj_1 {0}\\n".format(dv["dv_0"]))
    f.write("obj_2 {0}\\n".format((1.0 + dv["dv_1"]) / dv["dv_0"]))
"""

with open(os.path.join(workdir, "forward_run.py"), "w") as f:
    f.write(FORWARD_RUN)
with open(os.path.join(workdir, "dv.dat.tpl"), "w") as f:
    f.write("ptf ~\ndv_0 ~   dv_0   ~\ndv_1 ~   dv_1   ~\n")
with open(os.path.join(workdir, "dv.dat"), "w") as f:
    f.write("dv_0 0.5\ndv_1 2.5\n")
with open(os.path.join(workdir, "obj.dat.ins"), "w") as f:
    f.write("pif ~\nl1 w !obj_1!\nl1 w !obj_2!\n")
with open(os.path.join(workdir, "obj.dat"), "w") as f:
    f.write("obj_1 0.5\nobj_2 7.0\n")

# the control file, built with pyemu
here = os.getcwd()
os.chdir(workdir)
try:
    pst = pyemu.Pst.from_io_files("dv.dat.tpl", "dv.dat", "obj.dat.ins", "obj.dat")
finally:
    os.chdir(here)

par = pst.parameter_data
par.loc[:, ["partrans", "pargp", "parchglim"]] = ["none", "decvars", "relative"]
par.loc["dv_0", ["parlbnd", "parval1", "parubnd"]] = [0.1, 0.5, 1.0]
par.loc["dv_1", ["parlbnd", "parval1", "parubnd"]] = [0.0, 2.5, 5.0]

# both objectives are minimized, so both go in a less_than group
pst.observation_data.loc[:, ["weight", "obgnme"]] = [1.0, "less_than"]

# the two constraints, as prior information
pst.add_pi_equation(["dv_0", "dv_1"], pilbl="const_1", rhs=6.0,
                    coef_dict={"dv_0": 9.0, "dv_1": 1.0}, obs_group="greater_than")
pst.add_pi_equation(["dv_0", "dv_1"], pilbl="const_2", rhs=1.0,
                    coef_dict={"dv_0": 9.0, "dv_1": -1.0}, obs_group="greater_than")

pst.model_command = "python forward_run.py"
pst.pestpp_options["opt_dec_var_groups"] = "decvars"
pst.pestpp_options["mou_objectives"] = "obj_1,obj_2"
pst.pestpp_options["mou_population_size"] = POP
pst.pestpp_options["mou_generator"] = "de"
pst.pestpp_options["random_seed"] = 11
pst.control_data.noptmax = 20          # has to cover every generation we drive by hand
pst.write(os.path.join(workdir, "constr.pst"), version=2)
print(pst.prior_information.loc[:, ["pilbl", "equation", "obgnme"]].to_string(index=False))

## feasibility, in pandas

two helpers. `violation` is how far a member misses the two constraints, added up, and `repair`
pushes `dv_0` up until it stops missing them.

mou is perfectly happy to run infeasible members - it sorts them out afterwards through
constraint dominance, which is the right thing to do when feasibility costs a model run to
find out. here it doesnt, so every infeasible member is a run spent on a point we already know
is no good.

In [ ]:
def violation(dv):
    """how far each member misses the two constraints, added up. 0 means feasible"""
    v1 = np.minimum(0.0, 9.0 * dv["dv_0"] + dv["dv_1"] - 6.0)
    v2 = np.minimum(0.0, 9.0 * dv["dv_0"] - dv["dv_1"] - 1.0)
    return -(v1 + v2)


def n_infeasible(dv, tol=1.0e-10):
    return int((violation(dv) > tol).sum())


def repair(dv):
    """push dv_0 up to the smallest value that satisfies both constraints"""
    out = dv.copy()
    need = np.maximum((6.0 - out["dv_1"]) / 9.0, (1.0 + out["dv_1"]) / 9.0)
    out["dv_0"] = np.maximum(out["dv_0"], need).clip(0.1, 1.0)
    return out


def pareto_front(obs, dv):
    """the nondominated feasible members, as a frame"""
    feas = obs.loc[violation(dv) <= 1.0e-10, ["obj_1", "obj_2"]]
    pts = feas.values
    keep = [i for i, p in enumerate(pts)
            if not (((pts <= p).all(axis=1) & (pts < p).any(axis=1)).any())]
    return feas.iloc[keep].sort_values("obj_1")


def hypervolume(obs, dv, ref=(1.0, 10.0)):
    """area dominated by the feasible front, measured back to a reference corner.

    one number per generation that goes up as the front spreads out and moves in, so the
    generator schedule below has something to be judged against.
    """
    pts = pareto_front(obs, dv).values
    pts = pts[(pts[:, 0] < ref[0]) & (pts[:, 1] < ref[1])]
    hv, prev = 0.0, ref[1]
    for f1, f2 in pts:
        hv += (ref[0] - f1) * (prev - f2)
        prev = f2
    return hv

## start from a population you chose

`initialize(defer_runs=True)` draws the population and stops before running it, handing back how
many runs are waiting. the drawn population is sitting in `dv_pop()`, and whatever is in it when
`queue_runs()` is called is what gets run.

so this is where a repair operator, a latin hypercube design, a population from a previous study
or last year's calibration goes in.

In [ ]:
mou = Mou.from_pst("constr.pst", workdir=workdir)

n = mou.initialize(defer_runs=True)
print("runs waiting:", n)

drawn = mou.dv_pop(lower=True)
print("drawn population:", drawn.shape)
print("infeasible members:", n_infeasible(drawn), "of", drawn.shape[0])
drawn.head()

In [ ]:
mou.set_par_df(repair(drawn))
print("infeasible after repair:", n_infeasible(mou.dv_pop(lower=True)))

# now run the population we handed back, and let mou finish setting itself up
mou.queue_runs()
mou.run()
mou.process_runs()
mou.finish_initialize()

initial_dv = mou.dv_pop(lower=True)
initial_obs = mou.obs_pop(lower=True)
print("population size :", mou.population_size)
print("front members   :", pareto_front(initial_obs, initial_dv).shape[0])
print("hypervolume     : {0:.4g}".format(hypervolume(initial_obs, initial_dv)))
initial_obs.head()

## a different generator every generation

`mou_generator` is read at the point of use, once per generation, so it can change while the
tool is running:

| generator | what it is | good for |
|---|---|---|
| `de` | differential evolution | covering ground early |
| `sbx` | simulated binary crossover | recombining two parents |
| `pm` | polynomial mutation | small local moves, usually paired with `sbx` |
| `simplex` | nelder-mead style reflection | pushing on an existing front |

the option takes a list, and then each generation splits the offspring between the generators in
it. so `de,sbx,pm` is one generation bred three ways.

the schedule below is the usual explore-then-polish shape: de while the population is spread
out, sbx+pm once it has found the front, then all three together.

**pso is the exception.** it carries a velocity per member, keyed by member name, and those only
exist for members pso itself made. switch to pso partway through and the next generation dies
with `map::at: key not found`. if you want pso, use it for every generation.

In [ ]:
SCHEDULE = ["de"] * 3 + ["sbx,pm"] * 3 + ["de,sbx,pm"] * 3

history = [{"generation": 0, "generator": "(initial)",
            "feasible": POP - n_infeasible(initial_dv),
            "front": pareto_front(initial_obs, initial_dv).shape[0],
            "hypervolume": hypervolume(initial_obs, initial_dv)}]

for generator in SCHEDULE:
    mou.set_option("mou_generator", generator)      # <-- the whole switch
    step = mou.solve()

    dv, obs = mou.dv_pop(lower=True), mou.obs_pop(lower=True)
    history.append({"generation": step.iter, "generator": generator,
                    "feasible": mou.population_size - n_infeasible(dv),
                    "front": pareto_front(obs, dv).shape[0],
                    "hypervolume": hypervolume(obs, dv)})
    print("generation {0:2d}  {1:10s}  front {2:2d}  hypervolume {3:.4g}".format(
        step.iter, generator, history[-1]["front"], history[-1]["hypervolume"]))

history = pd.DataFrame(history)

## reaching into the offspring

`solve(defer_runs=True)` breeds the next generation and stops before running it. the offspring
are `candidates()` - one candidate population for mou, always, where ies has one per lambda x
scale factor.

a candidate is a handle onto the ensemble the run manager is about to be handed, not a copy of
it. `par_df()` reads it, and a write through `par_view()` is what gets run.

In [ ]:
mou.set_option("mou_generator", "de")
n = mou.solve(defer_runs=True)

cand = mou.candidates()[0]
kids = cand.par_df(lower=True)
print("offspring        :", kids.shape[0], "->", n, "runs")
print("infeasible       :", n_infeasible(kids))
kids.head()

In [ ]:
fixed = repair(kids)

# and while we are in here, two points we would like tried: both sit on the analytic front
# of this problem, where dv_1 = 6 - 9*dv_0
seeded = list(fixed.index[:2])
fixed.loc[seeded[0], ["dv_0", "dv_1"]] = [0.42, 6.0 - 9.0 * 0.42]
fixed.loc[seeded[1], ["dv_0", "dv_1"]] = [0.55, 6.0 - 9.0 * 0.55]

with cand.par_view() as view:
    np.asarray(view)[:, :] = fixed.values      # the write lands in the run

print("infeasible after repair:", n_infeasible(cand.par_df(lower=True)))
print("seeded members         :", seeded)

In [ ]:
mou.queue_runs()
mou.run()
mou.process_runs()
step = mou.finish_solve(defer_runs=True)

print("pending runs:", step.pending_runs, " (mou never needs a second batch)")
print("generation  :", step.iter)

dv, obs = mou.dv_pop(lower=True), mou.obs_pop(lower=True)
survived = [m for m in seeded if m in dv.index]
print("seeded members that survived selection:", survived)
print(obs.loc[survived])

history.loc[len(history)] = {"generation": step.iter, "generator": "de (by hand)",
                             "feasible": mou.population_size - n_infeasible(dv),
                             "front": pareto_front(obs, dv).shape[0],
                             "hypervolume": hypervolume(obs, dv)}

## what happened

the front on the left, the schedule on the right. the dashed line is the analytic front of
`constr`, which is `dv_1 = 6 - 9*dv_0` where that stays inside the bounds and `dv_1 = 0` after
that.

In [ ]:
final_dv, final_obs = mou.dv_pop(lower=True), mou.obs_pop(lower=True)

x = np.linspace(0.39, 1.0, 200)
true_f1 = x
true_f2 = np.where(x <= 6.0 / 9.0, (7.0 - 9.0 * x) / x, 1.0 / x)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ax = axes[0]
ax.plot(true_f1, true_f2, "k--", lw=1, label="analytic front")
ax.scatter(initial_obs["obj_1"], initial_obs["obj_2"], s=25, c="0.7",
           label="initial population")
ax.scatter(final_obs["obj_1"], final_obs["obj_2"], s=25, c="C0", label="final population")
front = pareto_front(final_obs, final_dv)
ax.plot(front["obj_1"], front["obj_2"], "C1o-", ms=5, lw=1, label="final front")
ax.set_xlabel("obj_1"), ax.set_ylabel("obj_2"), ax.set_ylim(0, 10)
ax.legend(loc="upper right", fontsize=8), ax.set_title("objective space")

ax = axes[1]
ax.plot(history["generation"], history["hypervolume"], "o-", c="C0")
for gen, grp in history.groupby("generator", sort=False):
    ax.scatter(grp["generation"], grp["hypervolume"], s=60, label=gen, zorder=3)
ax.set_xlabel("generation"), ax.set_ylabel("hypervolume")
ax.legend(loc="lower right", fontsize=8), ax.set_title("what each generator bought")

plt.tight_layout()

In [ ]:
print(history.to_string(index=False))

In [ ]:
mou.finalize()
mou.close()
print("done - the archive and the pareto summaries are in", workdir)
print([f for f in sorted(os.listdir(workdir)) if "pareto" in f or "archive" in f])

## notes and caveats

**the archive is not in the api.** mou keeps an archive of everything nondominated it has ever
seen, and that is written to `constr.pareto.archive.summary.csv` and
`constr.archive.dv_pop.csv` as it goes. what the api gives you is the live population, which is
the thing you can still change. read the archive off disk afterwards, or with
`pyemu.Results`.

**`pending_runs` is always 0 for mou.** ies runs a subset of the ensemble against every
candidate to pick a lambda and then has to run the rest at the winner, so its `finish_solve`
can hand back a second batch. mou breeds one population, runs it, and selects - one batch, so
the deferred loop is always just the one pass.

**noptmax still counts.** driving generations by hand doesnt exempt you from it. past noptmax
mou throws from inside the archive bookkeeping, so set it to cover however many generations you
intend to drive.

**this notebook is serial.** `Mou.from_pst(..., workers=8)` starts panther agents instead, and
nothing else in here changes.

**pso is the one generator that cannot be switched to mid run** - see the note above the
schedule. de, sbx, pm and simplex mix freely, in any order, changing every generation.

**repairing is not always the right call.** here feasibility is two lines of algebra. when it
takes a model run to find out whether a member is feasible, you cannot repair anything before
running it, and mou's constraint dominance is doing the only thing that can be done. the api
doesnt change that - it just gets out of the way when you do know something mou doesnt.